# Training AutoEncoder on simulated pathway states


In [78]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import os

# --- Config
USE_GLOBAL_ZSCORE = False
DRY_RUN = False #os.getenv("USER") == "polya"

# --- Device ---
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"User: {os.getenv('USER')} | Device: {device} | DRY_RUN: {DRY_RUN} | Global Z-score: {USE_GLOBAL_ZSCORE}")

User: polya | Device: mps | DRY_RUN: False | Global Z-score: False


# Wrangling data

In [79]:
import numpy as np
import pandas as pd
from notebooks.experiment.preprocessing import load_and_clean, make_windows_sample, DEFAULT_STIM_COLS

df = pd.read_parquet("dataset.parquet")

# Use all features (default), or pick a subset:
# stim_cols = ["u_t", "m_t", "recency"]
erk, stim, metadata = make_windows_sample(df, window_size=40, rng=42)

print(f"Windows: {erk.shape[0]}, Timepoints: {erk.shape[1]}")
print(f"Stim features: {len(DEFAULT_STIM_COLS)} channels — {DEFAULT_STIM_COLS}")
print(f"Stim shape: {stim.shape}")  # expect (n_windows, 9, 40)
print(f"\nSamples per experiment:")
print(metadata["ramp_pattern_name"].value_counts())

Windows: 11443, Timepoints: 40
Stim features: 9 channels — ['u_t', 'm_t', 'recency', 'ewma_fast', 'ewma_slow', 'n_5', 'slope_5', 'burst_pos', 's_cum']
Stim shape: (11443, 9, 40)

Samples per experiment:
ramp_pattern_name
3-2-1minIntervals    6118
Sustained            3659
ramp1                1300
Single                366
Name: count, dtype: int64
